In [3]:
import os
import math
import numpy as np
import pandas as pd

# =========================
# Config
# =========================
TARGET_COL = "Irrigation_Need"

CAT_COLS = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

# 必要なら後で追加
# CAT_COLS_WITH_BINARY = CAT_COLS + ["Mulching_Used"]

OUT_DIR = "/kaggle/working/high_iv"
os.makedirs(OUT_DIR, exist_ok=True)

EPS = 0.5  # zero division 回避用の smoothing


# =========================
# IV / WOE
# =========================
def calc_iv_for_categorical(
    df: pd.DataFrame,
    col: str,
    target_col: str,
    positive_label=1,
    eps: float = 0.5,
    treat_nan_as_str: bool = True,
) -> tuple[float, pd.DataFrame]:
    """
    二値 target 前提のカテゴリ列 IV を計算する。
    戻り値:
      iv_value: float
      detail_df: 各カテゴリごとの内訳
    """

    work = df[[col, target_col]].copy()

    if treat_nan_as_str:
        work[col] = work[col].astype("object").fillna("__MISSING__")
    else:
        work = work.dropna(subset=[col])

    # good / bad の定義
    # ここでは positive_label=1 を bad 側、0 を good 側として扱う
    # IV の大小には本質的影響なし
    bad_mask = (work[target_col] == positive_label)
    good_mask = ~bad_mask

    total_bad = bad_mask.sum()
    total_good = good_mask.sum()

    if total_bad == 0 or total_good == 0:
        raise ValueError("target が片側しかありません。IV を計算できません。")

    grouped = (
        work.groupby(col, dropna=False)[target_col]
        .agg(
            total_count="count",
            bad_count=lambda s: (s == positive_label).sum(),
            good_count=lambda s: (s != positive_label).sum(),
        )
        .reset_index()
        .rename(columns={col: "category"})
    )

    # smoothing
    grouped["bad_count_s"] = grouped["bad_count"] + eps
    grouped["good_count_s"] = grouped["good_count"] + eps

    bad_total_s = grouped["bad_count_s"].sum()
    good_total_s = grouped["good_count_s"].sum()

    grouped["bad_dist"] = grouped["bad_count_s"] / bad_total_s
    grouped["good_dist"] = grouped["good_count_s"] / good_total_s

    grouped["woe"] = np.log(grouped["good_dist"] / grouped["bad_dist"])
    grouped["iv_component"] = (grouped["good_dist"] - grouped["bad_dist"]) * grouped["woe"]

    iv_value = grouped["iv_component"].sum()

    grouped["bad_rate_in_category"] = grouped["bad_count"] / grouped["total_count"]
    grouped["share_of_rows"] = grouped["total_count"] / len(work)

    grouped = grouped.sort_values(
        ["iv_component", "total_count"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return float(iv_value), grouped


def calc_iv_table(
    df: pd.DataFrame,
    cat_cols: list[str],
    target_col: str,
    positive_label=1,
    eps: float = 0.5,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    rows = []
    detail_map = {}

    for col in cat_cols:
        iv_value, detail_df = calc_iv_for_categorical(
            df=df,
            col=col,
            target_col=target_col,
            positive_label=positive_label,
            eps=eps,
        )

        rows.append(
            {
                "feature": col,
                "iv": iv_value,
                "n_unique": df[col].nunique(dropna=False),
                "missing_count": df[col].isna().sum(),
            }
        )
        detail_map[col] = detail_df

    summary = pd.DataFrame(rows).sort_values("iv", ascending=False).reset_index(drop=True)
    summary["iv_rank"] = np.arange(1, len(summary) + 1)

    return summary, detail_map

In [4]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")

iv_summary, iv_detail_map = calc_iv_table(
    df=train,
    cat_cols=CAT_COLS,
    target_col=TARGET_COL,
    positive_label=1,
    eps=EPS,
)

display(iv_summary)

iv_summary.to_csv(f"{OUT_DIR}/iv_summary.csv", index=False)

for col, detail_df in iv_detail_map.items():
    detail_df.to_csv(f"{OUT_DIR}/iv_detail_{col}.csv", index=False)

ValueError: target が片側しかありません。IV を計算できません。

In [ ]:
TOP_K = 5
high_iv_cols = iv_summary.head(TOP_K)["feature"].tolist()

print("high_iv_cols =", high_iv_cols)